In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from huggingface_hub import HfApi, hf_hub_download
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.base import clone
from lightgbm import LGBMClassifier
import inspect
import os

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import pandas as pd
import geopandas as gpd
import os

# Hugging Face dataset repository identifier
HF_REPO_ID = "Loveffort/Capstone-dataset"
# Load token from environment variable, use static fallback if missing
HF_TOKEN = os.getenv("HF_TOKEN")
# Check if running on Google Colab environment
IS_COLAB = "COLAB_GPU" in os.environ
# Define local cache directory path
WORK_DIR = "/content/victoria_fire_risk_data" if IS_COLAB else "./victoria_fire_risk_data"
# Create working directory if it does not exist
os.makedirs(WORK_DIR, exist_ok=True)

class HFDataManager:
    def __init__(self, repo_id, token, work_dir):
        """
        Initialize manager for downloading/uploading files to Hugging Face dataset repo
        :param repo_id: Target HF dataset repository path
        :param token: HF authentication access token
        :param work_dir: Local directory for file cache
        """
        self.repo_id = repo_id
        self.token = token
        self.work_dir = work_dir
        # Initialize HF API client only when valid token exists
        self.api = HfApi(token=token) if token else None

    def load(self, hf_file_path):
        """
        Download file from HF Hub and parse into dataframe/geodataframe
        Supported formats: shp, parquet(geo/normal), geojson, csv, xlsx/xls
        :param hf_file_path: Relative file path inside HF repository
        :return: pd.DataFrame / gpd.GeoDataFrame
        """
        # Process multi-file shapefile dataset
        if hf_file_path.lower().endswith(".shp"):
            base = hf_file_path.replace(".shp", "")
            local_files = {}
            # Mandatory auxiliary extensions for shapefile
            ext_list = [".shp", ".shx", ".dbf", ".prj", ".cpg"]
            for ext in ext_list:
                filename = f"{base}{ext}"
                try:
                    local_file = hf_hub_download(
                        repo_id=self.repo_id,
                        repo_type="dataset",
                        filename=filename,
                        token=self.token,
                        cache_dir=self.work_dir
                    )
                    local_files[ext] = local_file
                except Exception as e:
                    print(f"Warning: {filename} not found ({e})")
            # Terminate execution if core .shp file missing
            shp_path = local_files.get(".shp")
            if not shp_path:
                raise FileNotFoundError("Shapefile .shp not found in repo")
            return gpd.read_file(shp_path)

        # Download single standalone file from repository
        local_file = hf_hub_download(
            repo_id=self.repo_id,
            repo_type="dataset",
            filename=hf_file_path,
            token=self.token,
            cache_dir=self.work_dir
        )

        # Parse file by extension type
        if hf_file_path.lower().endswith(".parquet"):
            # Prioritize GeoParquet to preserve geometry columns
            try:
                return gpd.read_parquet(local_file)
            except (ValueError, TypeError):
                # Fallback to standard parquet for non-spatial tables
                return pd.read_parquet(local_file)
        elif hf_file_path.lower().endswith(".geojson"):
            return gpd.read_file(local_file)
        elif hf_file_path.lower().endswith(".csv"):
            return pd.read_csv(local_file)
        elif hf_file_path.lower().endswith(".xlsx") or hf_file_path.lower().endswith(".xls"):
            return pd.read_excel(local_file)
        else:
            raise ValueError(f"Unsupported file format: {hf_file_path}")

    def save_gdf(self, gdf, file_name, format="parquet"):
        """
        Export dataframe/geodataframe to local disk then upload to HF dataset repo
        Differentiate handling for GeoDataFrame and standard DataFrame
        :param gdf: Input table (GeoDataFrame or DataFrame)
        :param file_name: Output filename without extension
        :param format: Export format: parquet / csv / geojson
        """
        if not self.api:
            raise ValueError("HF Token not configured!")

        file_fullname = f"{file_name}.{format}"
        local_path = os.path.join(self.work_dir, file_fullname)

        # Handle spatial GeoDataFrame
        if isinstance(gdf, gpd.GeoDataFrame):
            if format == "parquet":
                gdf.to_parquet(local_path, engine="pyarrow", index=False)
            elif format == "geojson":
                gdf.to_file(local_path, driver="GeoJSON")
            elif format == "csv":
                gdf.to_csv(local_path, index=False)
            else:
                raise ValueError("Unsupported file format for GeoDataFrame")
        # Handle non-spatial DataFrame
        else:
            if format == "parquet":
                gdf.to_parquet(local_path, engine="pyarrow", index=False)
            elif format == "csv":
                gdf.to_csv(local_path, index=False)
            else:
                raise ValueError("Unsupported file format for DataFrame")

        # Upload generated file to Hugging Face repository
        self.api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=file_fullname,
            repo_id=self.repo_id,
            repo_type="dataset",
            commit_message=f"Save {file_fullname}"
        )
        print(f"Uploaded to HF Hub: {self.repo_id}/{file_fullname}")

hf_manager = HFDataManager(HF_REPO_ID, HF_TOKEN, WORK_DIR)

In [ ]:
x_vector=hf_manager.load("x_vector.parquet")
x_vector_unnormalized=hf_manager.load("x_vector_unnormalized.parquet")
y_labels=hf_manager.load("y_labels.parquet")

In [ ]:
labeled_pfi = y_labels['PFI'].unique()
x_labeled = x_vector[x_vector['PFI'].isin(labeled_pfi)].copy()
x_unlabeled = x_vector[~x_vector['PFI'].isin(labeled_pfi)].copy()
y = y_labels['is_low_risk'].copy()

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    f1_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

# Feature preprocessing: log transform PFI to reduce skewness
x_features = x_labeled.copy()
x_features["PFI"] = np.log1p(x_features["PFI"])

# Stratified train/validation split to preserve class ratio
x_train, x_val, y_train, y_val = train_test_split(
    x_features, y, test_size=0.2, stratify=y, random_state=42
)

# Fixed hyperparameters optimized for small imbalanced dataset
params_fix = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.045,
    'n_estimators': 200,
    'max_depth': 3,
    'num_leaves': 8,
    'min_child_samples': 3,
    'subsample': 0.5,
    'colsample_bytree': 0.5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'verbose': -1,
    'scale_pos_weight': 2,
}

# Multi-seed training to select best stable model
seeds = [16,26,36,46,56,66,76,86]
best_auc = 0
best_model = None

for seed in seeds:
    model = lgb.LGBMClassifier(**params_fix, random_state=seed)
    model.fit(
        x_train, y_train,
        eval_set=[(x_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=0)],
        eval_metric='auc'
    )
    val_prob = model.predict_proba(x_val)[:,1]
    current_auc = roc_auc_score(y_val, val_prob)
    if current_auc > best_auc:
        best_auc = current_auc
        best_model = model

# Grid search for optimal classification threshold maximizing F1 score
y_prob = best_model.predict_proba(x_val)[:, 1]
best_thresh = 0.5
best_f1 = 0

for thresh in np.linspace(0.4, 0.7, 100):
    y_predict = (y_prob >= thresh).astype(int)
    current_f1 = f1_score(y_val, y_predict, zero_division=0)
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_thresh = thresh

y_pred = (y_prob >= best_thresh).astype(int)

# Print classification evaluation report
print("\n" + "="*60)
print("Initial Classification Report")
print("="*60)
print(classification_report(
    y_val, y_pred,
    target_names=['High Risk', 'Low Risk'],
    digits=4
))

# Plot confusion matrix heatmap
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['High Risk','Low Risk'],
            yticklabels=['High Risk','Low Risk'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# Initialize extended training set with original labeled data
x_extended = x_train.copy()
y_extended = y_train.copy()
current_model = best_model
max_iter = 3
new_samples_ratio_per_iter = 0.25

# Preprocess unlabeled feature set consistent with training data
x_unlabeled_features = x_unlabeled.copy()
x_unlabeled_features["PFI"] = np.log1p(x_unlabeled_features["PFI"])

# Iterative self-training loop
for i in range(max_iter):
    # Predict risk probability for all unlabeled samples
    unlabeled_pred = current_model.predict_proba(x_unlabeled_features)[:, 1]

    # Calculate number of new pseudo-label samples to add
    target_new_samples = int(len(x_extended) * new_samples_ratio_per_iter)
    k1 = int(target_new_samples * 0.5)
    k2 = target_new_samples - k1

    # Select extreme high/low confidence samples for pseudo labels
    low_risk_idx = np.argsort(unlabeled_pred)[:k2]
    high_risk_idx = np.argsort(unlabeled_pred)[-k1:]
    selected_idx = np.concatenate([low_risk_idx, high_risk_idx])
    selected_idx = np.unique(selected_idx)

    # Construct pseudo-label dataset
    new_x = x_unlabeled_features.iloc[selected_idx].copy()
    new_y = np.zeros(len(selected_idx))
    new_y[len(low_risk_idx):] = 1
    new_y = pd.Series(new_y, name='is_low_risk')

    # Append pseudo-label data to training pool
    x_extended = pd.concat([x_extended, new_x], ignore_index=True)
    y_extended = pd.concat([y_extended, new_y], ignore_index=True)

    # Retrain model on expanded dataset
    current_model = lgb.LGBMClassifier(**params_fix)
    current_model.fit(
        x_extended, y_extended,
        eval_set=[(x_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(30, verbose=0)]
    )

    # Remove samples already added to training set
    x_unlabeled_features = x_unlabeled_features.drop(new_x.index).reset_index(drop=True)
    val_auc = roc_auc_score(y_val, current_model.predict_proba(x_val)[:,1])

# Evaluate final semi-supervised model
print("\n" + "="*60)
print("Final Classification Report")
print("="*60)
val_prob = current_model.predict_proba(x_val)[:, 1]
val_pred = (val_prob >= best_thresh).astype(int)

print(classification_report(
    y_val, val_pred,
    target_names=['High Risk', 'Low Risk'],
    digits=4
))

# Plot final confusion matrix
cm = confusion_matrix(y_val, val_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['High Risk','Low Risk'],
            yticklabels=['High Risk','Low Risk'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Copy raw labeled feature data
df_labeled = x_labeled.copy()

# Preprocess features consistent with training input
model_input = df_labeled.copy()
model_input['PFI'] = np.log1p(model_input['PFI'])
model_input = model_input[x_features.columns]

# Generate low-risk probability predictions
df_labeled['risk_prob'] = np.round(current_model.predict_proba(model_input)[:, 1], 4)

# Merge ground truth risk labels matched by PFI
df_labeled = df_labeled.merge(y_labels[['PFI']], on='PFI', how='inner')
df_labeled['true_risk'] = df_labeled['PFI'].map(dict(zip(y_labels['PFI'], y_labels['is_low_risk'])))

# Split dataset by true risk category
df_low = df_labeled[df_labeled['true_risk'] == 1].copy()
df_high = df_labeled[df_labeled['true_risk'] == 0].copy()

# Linear scale low-risk samples to range [1, 10]
if not df_low.empty:
    score_base_low = 1 - df_low['risk_prob']
    df_low['risk_score'] = np.interp(score_base_low, (score_base_low.min(), score_base_low.max()), (1, 10))
    df_low['risk_level'] = 'low'

# Linear scale high-risk samples to range [86, 99]
if not df_high.empty:
    score_base_high = 1 - df_high['risk_prob']
    df_high['risk_score'] = np.interp(score_base_high, (score_base_high.min(), score_base_high.max()), (86, 99))
    df_high['risk_level'] = 'high'

# Combine labeled risk results
df_result = pd.concat([df_low, df_high], ignore_index=True)

# Clean output table sorted by PFI
risk_result = df_result[['PFI', 'risk_prob', 'risk_score', 'risk_level']].copy()
risk_result['risk_score'] = risk_result['risk_score'].round(2)
risk_result = risk_result.sort_values('PFI', ascending=True).reset_index(drop=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Copy raw unlabeled feature data
df_unlabeled = x_unlabeled.copy()

# Standard feature preprocessing
model_input = df_unlabeled.copy()
model_input['PFI'] = np.log1p(model_input['PFI'])
model_input = model_input[x_features.columns]

# Predict low-risk probability
df_unlabeled['risk_prob'] = np.round(current_model.predict_proba(model_input)[:, 1], 4)

# Sort samples by risk severity
df_unlabeled['sort_key'] = 1 - df_unlabeled['risk_prob']
df_unlabeled = df_unlabeled.sort_values(['sort_key', 'PFI'], ascending=[True, True]).reset_index(drop=True)

# Uniform linear mapping of all unlabeled samples to risk score range [1,99]
df_unlabeled['risk_score'] = np.linspace(1, 99, len(df_unlabeled))

# Classify risk tier by fixed score breakpoints
def assign_level(score):
    if score <= 10:
        return 'low'
    elif score <= 85:
        return 'medium'
    else:
        return 'high'

df_unlabeled['risk_level'] = df_unlabeled['risk_score'].apply(assign_level)

# Export clean result table
risk_result_unlabeled = df_unlabeled[['PFI', 'risk_prob', 'risk_score', 'risk_level']].copy()
risk_result_unlabeled['risk_score'] = risk_result_unlabeled['risk_score'].round(2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14, 6))

# Plot labeled low-risk distribution
if not risk_result[risk_result['risk_level'] == 'low'].empty:
    sns.kdeplot(
        risk_result[risk_result['risk_level'] == 'low']['risk_score'],
        label='Low Risk',
        fill=True,
        color='#2E8B57',
        alpha=0.6,
        linewidth=1.5
    )

# Plot labeled high-risk distribution
if not risk_result[risk_result['risk_level'] == 'high'].empty:
    sns.kdeplot(
        risk_result[risk_result['risk_level'] == 'high']['risk_score'],
        label='High Risk',
        fill=True,
        color='#DC143C',
        alpha=0.6,
        linewidth=1.5
    )

# Plot full unlabeled sample distribution
if not risk_result_unlabeled.empty:
    sns.kdeplot(
        risk_result_unlabeled['risk_score'],
        label='Unlabeled',
        fill=True,
        color='#4682B4',
        alpha=0.3,
        linewidth=1.5
    )

# Draw fixed risk tier threshold lines
plt.axvline(x=10, color='blue', linestyle='--', linewidth=2)
plt.axvline(x=85, color='blue', linestyle='--', linewidth=2)

# Chart formatting
plt.title('Risk Score Distribution (Labeled vs Unlabeled)', fontsize=14)
plt.xlabel('Risk Score', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.grid(alpha=0.3)
plt.legend(loc='upper right', fontsize=12)
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

In [ ]:
risk_result_all = pd.concat([risk_result, risk_result_unlabeled], ignore_index=True)
risk_result_all = risk_result_all.sort_values('PFI', ascending=True).reset_index(drop=True)

In [ ]:
hf_manager.save_gdf(risk_result_all, file_name="risk_result_all", format="parquet")

In [ ]:
feature_importance = pd.DataFrame({
    'feature': x_features.columns,
    'importance':current_model.feature_importances_
}).sort_values('importance', ascending=False).head(12)

print("="*60)
print("Top 12 Feature Importance ：")
print("="*60)
for idx, row in feature_importance.iterrows():
    print(f"{row['feature']:<30} Importance: {row['importance']:>5d}")

In [ ]:
risk_result_all=hf_manager.load("risk_result_all.parquet")
station_property_gdf=hf_manager.load("station_property_gdf.parquet")
substation_gdf=hf_manager.load("substation_gdf.parquet")
x_vector_unnormalized=hf_manager.load("x_vector_unnormalized.parquet")

In [ ]:
# Step 1: Merge full risk results with raw unnormalized feature vectors
risk_result_all['PFI'] = risk_result_all['PFI'].astype(int)
x_vector_unnormalized['PFI'] = x_vector_unnormalized['PFI'].astype(int)
df_step4_1 = pd.merge(
    risk_result_all,
    x_vector_unnormalized,
    on='PFI',
    how='left'
)

# Step 2: Merge with substation-property spatial mapping to retain polygon geometry
station_property_gdf['PFI'] = station_property_gdf['PFI'].astype(int)
df_step4_2 = pd.merge(
    df_step4_1,
    station_property_gdf[['PFI', 'substation', 'geometry']],
    on='PFI',
    how='left'
)

# Step3: Merge auxiliary substation attributes, drop duplicate geometry column
substation_attr = substation_gdf.drop(columns=['geometry'], errors='ignore')
substation_attr['substation'] = substation_attr['substation'].astype(str)
df_step4_2['substation'] = df_step4_2['substation'].astype(str)

df_final = pd.merge(
    df_step4_2,
    substation_attr,
    on='substation',
    how='left'
)

# Flip risk probability for intuitive high-risk representation
df_final['risk_prob'] = 1 - df_final['risk_prob']

# Convert combined table to valid GeoDataFrame with property polygon geometry
df_final = gpd.GeoDataFrame(
    df_final,
    geometry='geometry',
    crs=station_property_gdf.crs
)

# Upload final spatial dataset to Hugging Face repository
hf_manager.save_gdf(df_final, file_name="final_result", format="parquet")

In [ ]:
df_final=hf_manager.load("final_result.parquet")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.figure(figsize=(16, 6))

# Left subplot: curtailment rate distribution comparison
plt.subplot(1, 2, 1)
sns.kdeplot(df_final["cur_rate"], fill=True, color="#3274A1", alpha=0.6, linewidth=2, label="cur_rate")
sns.kdeplot(df_final["cur_rate25"], fill=True, color="#E1814C", alpha=0.6, linewidth=2, label="cur_rate25")
plt.xlim(0, 2.0)
plt.title("Distribution of cur_rate vs cur_rate25", fontsize=14, pad=12)
plt.xlabel("cur_rate")
plt.ylabel("Density")
plt.grid(alpha=0.2)
plt.legend()

# Right subplot: total curtailment energy distribution comparison
plt.subplot(1, 2, 2)
sns.kdeplot(df_final["curtailment"], fill=True, color="#3274A1", alpha=0.6, linewidth=2, label="curtailment")
sns.kdeplot(df_final["curtailment_25"], fill=True, color="#E1814C", alpha=0.6, linewidth=2, label="curtailment_25")
plt.xlim(0, 200000)
plt.title("Distribution of curtailment vs curtailment_25", fontsize=14, pad=12)
plt.xlabel("curtailment (KWh)")
plt.ylabel("Density")
plt.grid(alpha=0.2)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
pip install contextily

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import contextily as cx
from matplotlib.patches import Patch

# Random sample subset to reduce plotting computation load
df_sample = df_final.sample(n=1000000, random_state=42).copy()
gdf_sample = gpd.GeoDataFrame(df_sample, geometry="geometry", crs=df_final.crs)

# Quantile-based risk tier color segmentation
q1, q2 = gdf_sample["risk_score"].quantile([0.33, 0.66])
bins = [gdf_sample["risk_score"].min() - 1, q1, q2, gdf_sample["risk_score"].max()]
labels = ["low", "medium", "high"]
colors = ["#2ecc71", "#f1c40f", "#e74c3c"]
color_map = dict(zip(labels, colors))

gdf_sample["risk_level"] = pd.cut(gdf_sample["risk_score"], bins=bins, labels=labels)
gdf_sample["color_code"] = gdf_sample["risk_level"].map(color_map)

# Initialize high-resolution plotting canvas
fig, ax = plt.subplots(figsize=(20, 20), dpi=300)

# Plot property polygons with risk color coding
gdf_sample.plot(
    ax=ax,
    color=gdf_sample["color_code"],
    edgecolor="none",
    linewidth=0,
    alpha=0.9,
)

# Custom risk level legend
legend_elements = [
    Patch(facecolor="#2ecc71", label="Low "),
    Patch(facecolor="#f1c40f", label="Medium "),
    Patch(facecolor="#e74c3c", label="High ")
]
ax.legend(
    handles=legend_elements,
    loc="lower right",
    fontsize=16,
    title="Risk Level",
    title_fontsize=18,
    facecolor="white",
    edgecolor="black"
)

# Add lightweight basemap background
cx.add_basemap(
    ax,
    crs=gdf_sample.crs.to_string(),
    source=cx.providers.CartoDB.Positron,
    alpha=1.0
)

# Auto-fit axis bounds to sampled geometry range
minx, miny, maxx, maxy = gdf_sample.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)

# Remove axis ticks and borders for clean output
ax.set_axis_off()
ax.set_title(
    "Victoria Property Fire Risk Score (Sampled 1,000,000)",
    fontsize=22,
    pad=30,
    weight='bold'
)

plt.tight_layout()
plt.show()